In [ ]:
using Base.Threads
println( "Number of threads: ", nthreads() )

include( "../args.jl" )
include( "../model.jl" )
include( "../geom.jl" )
include( "../recur.jl" )

base = "../data/results/";

In [ ]:
# Case and data folder, default parameters.
N = 100
μ = 1.0
ξ = 0.1;  ϕ = 0.25
ρ = 0.1

# Import default scale.
dparams = Params()
dscale = Scale( 1.0, ξ )
dnondim = Nondim( dparams; scale=dscale )

# Temporal variables.
T = 5000
tlist = 0:5:T

# Range for density values.
μmin = -1;  μmax = 1
Nμ = 51;  Δμ = (μmax - μmin)/(Nμ-1)
μlist = round.( 10.0.^(μmin:Δμ:μmax), digits=6 )  # NON-DIMENSIONAL

# Adapatable time-step length.
smin = -3;  smax = 1
Ns = Nμ;  Δs = (smax - smin)/(Ns-1)
slist = round.( 10.0.^(smin:Δs:smax), digits=6 )  # NON-DIMENSIONAL

# Number of parameter combinations.
println( "Imported simulation data for $(Nμ) different values of μ." )
println( "Imported simulation data for $(Ns) different values of s0." )

In [ ]:
# Non-dimensional parameter list.
nondimlist = [Nondim( adjparams( dparams; ρ=ρ, s=s/ξ^(ϕ + 1) ); scale=dscale ) for s ∈ slist]

# Data folder name.
folderdata = [[findfolder( N, μ, nondim; base=base ) for nondim ∈ nondimlist] for μ ∈ μlist];

In [ ]:
# Import activity and determinism data.
M = 10
adatadata = [[readdlm( folder*"activity_T-$(round( defInt, T ))_M-$(M).txt" )
    for folder ∈ folderlist] for folderlist ∈ folderdata]
ςdatalist = [hcat( [readdlm( folder*"determinism_T-$(round( defInt, T ))_M-$(M).txt" ) for folder ∈ folderlist]... )
    for folderlist ∈ folderdata];

In [ ]:
# Determinism statistics.
ς̄data = [vcat( mean( ςdata, dims=1 )... ) for ςdata ∈ ςdatalist];

In [ ]:
# Subset folder.
Nlist = [100,1000];  M̂ = 50
ŝlist = round.( 10.0.^(-3:0.25:1), digits=6 );  Nŝ = length( ŝlist )
subsetdata = [[findfolder( N̂, μ, Nondim( adjparams( dparams; ρ=ρ, s=s/ξ^(ϕ + 1) ); scale=dscale ); base=base )
    for s ∈ ŝlist] for N̂ ∈ Nlist]

# Import determinism under default parameter case.
ς̂datalist = [hcat( [readdlm( folder*"determinism_T-$(round( defInt, T ))_M-$(M̂).txt" ) for folder ∈ subsetlist]... )
    for subsetlist ∈ subsetdata]

# Compute mean and standard devation.
μς̂data = hcat( [vcat( mean( ς̂data, dims=1 )... ) for ς̂data ∈ ς̂datalist]... )
σς̂data = hcat( [vcat( std(  ς̂data, dims=1 )... ) for ς̂data ∈ ς̂datalist]... )

# Compute critical parameter estimates.
μc = criticalμ( dnondim, N )

In [ ]:
# Plot the determinism in each case for visual inspection.
plt = plot(; size=(300,200), dpi=100 )

for (k, ς̄list) ∈ enumerate( ς̄data )
    plot!( plt, slist, ς̄list; lw=2, marker=:circ, label="" )
end

plot!( plt; xlims=(slist[1],slist[end]), xscale=:log10 )
plot!( plt; ylims=(0,1) )

plot!( plt; xlabel="effective temperature, "*L"s_0", ylabel="determinism", legend=:outerright )

In [ ]:
cmap = cgrad( :ice, [1, 2]./3, rev=false )

# Generate heat map of determinism as a function of ρ and β.
plt = plot(; size=(300,275), dpi=100 )
plot!( plt; left_margin=0pt, bottom_margin=-5pt, right_margin=15pt )

# Plot the density-dependent phase transition.
heatmap!( plt, slist, μlist, hcat( ς̄data... )'; cmap=cmap, clims=(0,1), colorbar=false )
plot!( plt, [slist[1],slist[end]], [μc, μc]; color=:gray68, lw=3, label="critical "*L"μ" )

plot!( plt; xlims=(slist[1],slist[end]), xscale=:log10 )
plot!( plt; ylims=(μlist[1],μlist[end]), yscale=:log10 )

plot!( plt; xlabel="speed constant, "*L"s_0", ylabel="density, "*L"μ", legend=:topright )

# saveplot( plt, figurefolder*"determinism-vs-spd_mu.png"; background=:white, dpi=600 )

In [ ]:
# Create dummy gradient data.
N = 1000
vals = reshape(range(0, 1, length=N), 1, :)

# Plot the gradient to make colorbar.
plt = plot(; size=(300,55), dpi=100 )

plot!( plt; left_margin=38.75pt, right_margin=15.25pt, top_margin=0pt, bottom_margin=15pt )

heatmap!( plt, vals; c=cmap, clims=(0,1),
    colorbar=false, ytick=false, yticks=false,
    framestyle=:box,
    xticks=( [1, round(Int,N/4), round(Int,N/2), round(Int,3N/4), N],
              ["0.0","0.25","0.50","0.75","1.0"] ) )

plot!( plt; xlabel="determinism, "*L"ς" )

# saveplot( plt, figurefolder*"determinism-vs-spd_mu_colorbar.png"; background=:white, dpi=600 );

In [ ]:
cmap = cgrad( :ice, [1, 2]./3, rev=false )

# Generate heat map of determinism as a function of ρ and β.
plt = plot(; layout=@layout([a; b{0.6h}]), size=(275,475), dpi=100 )
plot!( plt; left_margin=5pt, bottom_margin=-5pt, right_margin=15pt )

# Plot the temperature-dependent phase transition.
plot!( plt[1], ŝlist, μς̂data; ribbon=σς̂data, color=[:black :cornflowerblue], fillalpha=[1/4 1/2], lw=2, marker=:circ,
    label=hcat( [latexstring( "N=$(N̂)" ) for N̂ ∈ Nlist]... ) )

plot!( plt[1]; xlims=(slist[1],slist[end]), xscale=:log10 )
plot!( plt[1]; ylims=(0,1) )

title!( plt[1], "(a)"; title_position=:left )

plot!( plt[1]; xlabel="speed constant, "*L"s_0", ylabel="determinism, "*L"ς"*"\n", legend=:bottomright )

# Plot the density-dependent phase transition.
heatmap!( plt[2], slist, μlist, hcat( ς̄data... )'; cmap=cmap, clims=(0,1), colorbar=false )
plot!( plt[2], [slist[1],slist[end]], [μc, μc]; color=:gray68, lw=3, label="critical "*L"μ" )

plot!( plt[2]; xlims=(slist[1],slist[end]), xscale=:log10 )
plot!( plt[2]; ylims=(μlist[1],μlist[end]), yscale=:log10 )

title!( plt[2], "(b)"; title_position=:left )

plot!( plt[2]; xlabel="speed constant, "*L"s_0", ylabel="density, "*L"μ", legend=:topright )

# saveplot( plt, figurefolder*"determinism-vs-spd_mu.png"; background=:white, dpi=600 )